# Retriever Testing Notebook

This notebook implements and tests the **retrieval system** for our RAG pipeline.

**What it does:**
1. Loads the pre-built FAISS HNSW index and ChromaDB collection
2. Rebuilds the metadata mapping (needed for FAISS since it only stores vectors)
3. Embeds user queries using the same `all-MiniLM-L6-v2` model
4. Applies **metadata pre-filters** (subject, class, content_type, etc.)
5. Executes HNSW nearest-neighbor search on **both** backends
6. Benchmarks and compares FAISS vs ChromaDB for latency and result quality

## Cell 1 - Install Dependencies

In [ ]:
!pip install faiss-cpu chromadb sentence-transformers pandas tqdm matplotlib numpy

## Cell 2 - Imports

In [ ]:
import json
import os
import time
import numpy as np
import pandas as pd
import faiss
import chromadb
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
from collections import defaultdict

print("All imports successful!")

## Cell 3 - Reload Document Metadata

FAISS only stores vectors (no metadata). We need to rebuild the exact same
`documents`, `metadatas`, and `ids` lists so that FAISS index positions
map correctly back to the original text and metadata.

**Important:** This must use the exact same loading order as `indexer_testing.ipynb`.

In [ ]:
DATASET_DIR = r'c:\Users\kaushal\Desktop\major 2\dataset'
RAG_CHUNKS_FILE = os.path.join(DATASET_DIR, 'rag_chunks', 'all_chunks.jsonl')
STRUCTURED_DIR = os.path.join(DATASET_DIR, 'structured')

documents = []
metadatas = []
ids = []

# 1. Load Pre-chunked RAG data (same order as indexer)
print(f"Loading data from {RAG_CHUNKS_FILE}...")
if os.path.exists(RAG_CHUNKS_FILE):
    with open(RAG_CHUNKS_FILE, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if not line.strip():
                continue
            data = json.loads(line)
            text = data.get('text', '')
            chunk_id = data.get('chunk_id', f'chunk_{i}')
            if text:
                documents.append(text)
                metadata = {k: str(v) if v is not None else '' for k, v in data.items() if k != 'text'}
                metadatas.append(metadata)
                ids.append(chunk_id)

print(f"Loaded {len(documents)} documents from all_chunks.jsonl")

# 2. Load Structured JSONs (same order as indexer)
print(f"Scanning for structured JSON files in {STRUCTURED_DIR}...")
for root, dirs, files in os.walk(STRUCTURED_DIR):
    for file in files:
        if file.endswith('.json'):
            file_path = os.path.join(root, file)
            with open(file_path, 'r', encoding='utf-8') as f:
                try:
                    data = json.load(f)
                    if isinstance(data, list):
                        for idx, item in enumerate(data):
                            text = item.get('text', '')
                            if text:
                                documents.append(text)
                                metadata = {k: str(v) if v is not None else '' for k, v in item.items() if k != 'text'}
                                metadata['source_file'] = file_path
                                metadatas.append(metadata)
                                ids.append(f"{file}_{idx}")
                except json.JSONDecodeError:
                    print(f"Error decoding JSON from {file_path}")

print(f"Total documents loaded: {len(documents)}")
print(f"\nSample metadata keys: {list(metadatas[0].keys())}")
print(f"Available subjects: {set(m.get('subject', 'N/A') for m in metadatas)}")
print(f"Available classes: {set(m.get('class', 'N/A') for m in metadatas)}")
print(f"Available content_types: {set(m.get('content_type', 'N/A') for m in metadatas)}")

## Cell 4 - Load Pre-Built Indices & Embedding Model

In [ ]:
# ---- Load Embedding Model ----
print("Loading embedding model (all-MiniLM-L6-v2)...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# ---- Load FAISS HNSW Index ----
FAISS_INDEX_PATH = './faiss_hnsw_index.bin'
print(f"Loading FAISS index from {FAISS_INDEX_PATH}...")
faiss_index = faiss.read_index(FAISS_INDEX_PATH)
print(f"FAISS index loaded: {faiss_index.ntotal} vectors, dimension={faiss_index.d}")

# ---- Load ChromaDB Collection ----
CHROMA_DB_PATH = './chroma_db'
print(f"Loading ChromaDB from {CHROMA_DB_PATH}...")
chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
collection = chroma_client.get_collection(name="rag_collection")
print(f"ChromaDB collection loaded: {collection.count()} vectors")

# ---- Sanity check ----
assert faiss_index.ntotal == len(documents), \
    f"FAISS vector count ({faiss_index.ntotal}) != document count ({len(documents)}). Metadata mapping will be wrong!"
assert collection.count() == len(documents), \
    f"ChromaDB count ({collection.count()}) != document count ({len(documents)})."

print(f"\n[OK] All loaded successfully! {len(documents)} documents across both indices.")

## Cell 5 - Retrieval Functions

Core retrieval logic for both backends:

| Backend | Metadata Filtering | Strategy |
|---------|-------------------|----------|
| **FAISS** | Post-retrieval | Over-fetch k * overfetch_factor, then filter by metadata |
| **ChromaDB** | Native where clause | Built-in pre-filter before HNSW search |

In [ ]:
def embed_query(query: str) -> np.ndarray:
    """Encode a single query string into a 384-dim embedding vector."""
    return model.encode([query])


def build_chroma_where_filter(filters: dict) -> dict:
    """
    Convert a simple {field: value} dict into a ChromaDB 'where' clause.
    
    Single filter:  {"subject": "Physics"} -> {"subject": "Physics"}
    Multi-filter:   {"subject": "Physics", "class": "12"} -> {"$and": [{...}, {...}]}
    """
    if not filters:
        return None
    
    conditions = [{key: {"$eq": value}} for key, value in filters.items()]
    
    if len(conditions) == 1:
        return conditions[0]
    return {"$and": conditions}


def check_metadata_match(metadata: dict, filters: dict) -> bool:
    """Check if a metadata dict matches all filter criteria."""
    for key, value in filters.items():
        if metadata.get(key, '') != value:
            return False
    return True


def search_faiss(query: str, k: int = 5, filters: dict = None, overfetch_factor: int = 10):
    """
    Search FAISS HNSW index with optional post-retrieval metadata filtering.
    
    Args:
        query: User question
        k: Number of results to return
        filters: Dict of {metadata_field: value} for post-filtering
        overfetch_factor: Fetch k * this many results before filtering
    
    Returns:
        dict with keys: documents, metadatas, ids, distances, latency_ms
    """
    start_time = time.perf_counter()
    
    # Step 1: Embed the query
    query_embedding = embed_query(query)
    embed_time = time.perf_counter()
    
    # Step 2: Search FAISS (over-fetch if filtering)
    fetch_k = k * overfetch_factor if filters else k
    fetch_k = min(fetch_k, faiss_index.ntotal)  # Don't exceed index size
    
    D, I = faiss_index.search(query_embedding, k=fetch_k)
    search_time = time.perf_counter()
    
    # Step 3: Post-filter by metadata
    result_docs = []
    result_metas = []
    result_ids = []
    result_dists = []
    
    for dist, idx in zip(D[0], I[0]):
        if idx < 0 or idx >= len(documents):  # FAISS returns -1 for missing
            continue
        
        if filters and not check_metadata_match(metadatas[idx], filters):
            continue
        
        result_docs.append(documents[idx])
        result_metas.append(metadatas[idx])
        result_ids.append(ids[idx])
        result_dists.append(float(dist))
        
        if len(result_docs) >= k:
            break
    
    end_time = time.perf_counter()
    
    return {
        'documents': result_docs,
        'metadatas': result_metas,
        'ids': result_ids,
        'distances': result_dists,
        'latency_ms': {
            'embedding': (embed_time - start_time) * 1000,
            'search': (search_time - embed_time) * 1000,
            'filtering': (end_time - search_time) * 1000,
            'total': (end_time - start_time) * 1000,
        },
        'total_fetched': fetch_k,
        'results_after_filter': len(result_docs),
    }


def search_chroma(query: str, k: int = 5, filters: dict = None):
    """
    Search ChromaDB collection with native metadata pre-filtering.
    
    Args:
        query: User question
        k: Number of results to return
        filters: Dict of {metadata_field: value} for native where-clause filtering
    
    Returns:
        dict with keys: documents, metadatas, ids, distances, latency_ms
    """
    start_time = time.perf_counter()
    
    # Step 1: Embed the query
    query_embedding = embed_query(query)
    embed_time = time.perf_counter()
    
    # Step 2: Build where filter and query ChromaDB
    where_filter = build_chroma_where_filter(filters)
    
    query_params = {
        'query_embeddings': query_embedding.tolist(),
        'n_results': k,
        'include': ['documents', 'metadatas', 'distances'],
    }
    if where_filter:
        query_params['where'] = where_filter
    
    results = collection.query(**query_params)
    search_time = time.perf_counter()
    
    end_time = time.perf_counter()
    
    return {
        'documents': results['documents'][0] if results['documents'] else [],
        'metadatas': results['metadatas'][0] if results['metadatas'] else [],
        'ids': results['ids'][0] if results['ids'] else [],
        'distances': results['distances'][0] if results['distances'] else [],
        'latency_ms': {
            'embedding': (embed_time - start_time) * 1000,
            'search': (search_time - embed_time) * 1000,
            'total': (end_time - start_time) * 1000,
        },
    }


def search_both(query: str, k: int = 5, filters: dict = None):
    """
    Run retrieval on both FAISS and ChromaDB and return combined results.
    """
    faiss_results = search_faiss(query, k=k, filters=filters)
    chroma_results = search_chroma(query, k=k, filters=filters)
    
    return {
        'faiss': faiss_results,
        'chroma': chroma_results,
    }


def display_results(results: dict, backend_name: str, max_text_len: int = 200):
    """Pretty-print search results from a single backend."""
    print(f"\n{'='*80}")
    print(f"  {backend_name} Results")
    print(f"{'='*80}")
    print(f"  Latency: {results['latency_ms']['total']:.2f} ms total")
    for key, val in results['latency_ms'].items():
        if key != 'total':
            print(f"    |-- {key}: {val:.2f} ms")
    print(f"  Results returned: {len(results['documents'])}")
    print(f"{'-'*80}")
    
    for i, (doc, meta, dist) in enumerate(zip(
        results['documents'], results['metadatas'], results['distances']
    )):
        subject = meta.get('subject', 'N/A')
        cls = meta.get('class', 'N/A')
        ctype = meta.get('content_type', 'N/A')
        chapter = meta.get('chapter_name', 'N/A')
        
        print(f"\n  Rank {i+1} | Distance: {dist:.4f}")
        print(f"  Subject: {subject} | Class: {cls} | Type: {ctype}")
        print(f"  Chapter: {chapter[:60]}")
        print(f"  Text: {doc[:max_text_len]}...")
    print()


print("[OK] Retrieval functions defined!")

---
## Cell 6 - Test 1: Basic Unfiltered Query

Search both backends without any metadata filters.

In [ ]:
query = "What are the phases of the author's relationship with his grandmother?"
print(f"Query: {query}\n")

results = search_both(query, k=5)

display_results(results['faiss'], 'FAISS HNSW')
display_results(results['chroma'], 'ChromaDB HNSW')

## Cell 7 - Test 2: Filtered Query (Subject + Class)

Filter results to only **Physics, Class 12** content.

In [ ]:
query = "What is Coulomb's law and how does it relate to electric force?"
filters = {"subject": "Physics", "class": "12"}

print(f"Query: {query}")
print(f"Filters: {filters}\n")

results = search_both(query, k=5, filters=filters)

display_results(results['faiss'], 'FAISS HNSW (Physics, Class 12)')
display_results(results['chroma'], 'ChromaDB HNSW (Physics, Class 12)')

## Cell 8 - Test 3: Filtered by Content Type

Only return **definitions** from **Chemistry**.

In [ ]:
query = "What is a solution and what are the types of solutions?"
filters = {"subject": "Chemistry", "content_type": "definition"}

print(f"Query: {query}")
print(f"Filters: {filters}\n")

results = search_both(query, k=5, filters=filters)

display_results(results['faiss'], 'FAISS (Chemistry Definitions)')
display_results(results['chroma'], 'ChromaDB (Chemistry Definitions)')

---
## Cell 9 - Benchmark: Latency Comparison (FAISS vs ChromaDB)

Run multiple queries through both backends and measure latency statistics.
We time:
- **Embedding** time (should be identical for both)
- **Search** time (the actual HNSW lookup)
- **Total** time (end-to-end)

In [ ]:
# Benchmark queries covering different subjects and difficulty
benchmark_queries = [
    # Physics
    {"query": "Explain Newton's second law of motion", "filters": None},
    {"query": "What is electromagnetic induction?", "filters": {"subject": "Physics"}},
    {"query": "Derive the expression for electric field due to a dipole", "filters": {"subject": "Physics", "class": "12"}},
    # Chemistry
    {"query": "What are the colligative properties of solutions?", "filters": None},
    {"query": "Explain the band theory of solids", "filters": {"subject": "Chemistry"}},
    {"query": "What is electrochemistry?", "filters": {"subject": "Chemistry", "class": "12"}},
    # English
    {"query": "Describe the portrait of the grandmother", "filters": {"subject": "English"}},
    {"query": "What is the central theme of the poem?", "filters": None},
    # Science (Class 9)
    {"query": "What is the structure of an atom?", "filters": {"subject": "Science"}},
    {"query": "Describe the classification of living organisms", "filters": None},
    # Content type filters
    {"query": "Define Ohm's law", "filters": {"content_type": "definition"}},
    {"query": "Solve the numerical problem on resistance", "filters": {"content_type": "exercise"}},
]

NUM_RUNS = 3  # Repeat each query for stable measurements
K = 5

benchmark_results = []

print(f"Running benchmark: {len(benchmark_queries)} queries x {NUM_RUNS} runs each...\n")

for q_info in benchmark_queries:
    query = q_info['query']
    filters = q_info['filters']
    
    for run in range(NUM_RUNS):
        # FAISS
        faiss_result = search_faiss(query, k=K, filters=filters)
        # ChromaDB
        chroma_result = search_chroma(query, k=K, filters=filters)
        
        benchmark_results.append({
            'query': query[:50],
            'filters': str(filters) if filters else 'None',
            'run': run + 1,
            'faiss_embed_ms': faiss_result['latency_ms']['embedding'],
            'faiss_search_ms': faiss_result['latency_ms']['search'],
            'faiss_total_ms': faiss_result['latency_ms']['total'],
            'chroma_embed_ms': chroma_result['latency_ms']['embedding'],
            'chroma_search_ms': chroma_result['latency_ms']['search'],
            'chroma_total_ms': chroma_result['latency_ms']['total'],
            'faiss_results_count': len(faiss_result['documents']),
            'chroma_results_count': len(chroma_result['documents']),
        })

df_bench = pd.DataFrame(benchmark_results)
print("[OK] Benchmark complete!")
print(f"\nTotal measurements: {len(df_bench)}")
df_bench.head(10)

## Cell 10 - Benchmark Results: Summary Statistics

In [ ]:
# Aggregate stats
summary = pd.DataFrame({
    'Metric': [
        'Embedding (ms)', 'HNSW Search (ms)', 'Total (ms)',
    ],
    'FAISS Mean': [
        df_bench['faiss_embed_ms'].mean(),
        df_bench['faiss_search_ms'].mean(),
        df_bench['faiss_total_ms'].mean(),
    ],
    'FAISS Median': [
        df_bench['faiss_embed_ms'].median(),
        df_bench['faiss_search_ms'].median(),
        df_bench['faiss_total_ms'].median(),
    ],
    'FAISS P95': [
        df_bench['faiss_embed_ms'].quantile(0.95),
        df_bench['faiss_search_ms'].quantile(0.95),
        df_bench['faiss_total_ms'].quantile(0.95),
    ],
    'Chroma Mean': [
        df_bench['chroma_embed_ms'].mean(),
        df_bench['chroma_search_ms'].mean(),
        df_bench['chroma_total_ms'].mean(),
    ],
    'Chroma Median': [
        df_bench['chroma_embed_ms'].median(),
        df_bench['chroma_search_ms'].median(),
        df_bench['chroma_total_ms'].median(),
    ],
    'Chroma P95': [
        df_bench['chroma_embed_ms'].quantile(0.95),
        df_bench['chroma_search_ms'].quantile(0.95),
        df_bench['chroma_total_ms'].quantile(0.95),
    ],
})

# Round for readability
for col in summary.columns[1:]:
    summary[col] = summary[col].round(2)

print("\n" + "="*90)
print("  BENCHMARK SUMMARY: FAISS HNSW vs ChromaDB HNSW")
print("="*90)
print(summary.to_string(index=False))

# Determine winner
faiss_avg = df_bench['faiss_search_ms'].mean()
chroma_avg = df_bench['chroma_search_ms'].mean()
speedup = chroma_avg / faiss_avg if faiss_avg > 0 else 0

print(f"\n{'-'*90}")
if faiss_avg < chroma_avg:
    print(f"  >>> FAISS is {speedup:.1f}x faster for HNSW search")
    print(f"      FAISS avg search: {faiss_avg:.2f} ms  |  ChromaDB avg search: {chroma_avg:.2f} ms")
else:
    print(f"  >>> ChromaDB is {1/speedup:.1f}x faster for HNSW search")
    print(f"      ChromaDB avg search: {chroma_avg:.2f} ms  |  FAISS avg search: {faiss_avg:.2f} ms")
print(f"{'-'*90}")

## Cell 11 - Visualization: Latency Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('FAISS HNSW vs ChromaDB HNSW - Latency Comparison', fontsize=14, fontweight='bold')

# --- Plot 1: Search Latency Distribution ---
ax1 = axes[0]
bp = ax1.boxplot(
    [df_bench['faiss_search_ms'], df_bench['chroma_search_ms']],
    labels=['FAISS', 'ChromaDB'],
    patch_artist=True,
)
bp['boxes'][0].set_facecolor('#4A90D9')
bp['boxes'][0].set_alpha(0.7)
bp['boxes'][1].set_facecolor('#50C878')
bp['boxes'][1].set_alpha(0.7)
ax1.set_title('HNSW Search Latency')
ax1.set_ylabel('Latency (ms)')
ax1.grid(axis='y', alpha=0.3)

# --- Plot 2: Total Latency per Query ---
ax2 = axes[1]
faiss_per_query = df_bench.groupby('query')['faiss_total_ms'].mean()
chroma_per_query = df_bench.groupby('query')['chroma_total_ms'].mean()

x = np.arange(len(faiss_per_query))
width = 0.35
ax2.bar(x - width/2, faiss_per_query.values, width, label='FAISS', color='#4A90D9', alpha=0.8)
ax2.bar(x + width/2, chroma_per_query.values, width, label='ChromaDB', color='#50C878', alpha=0.8)
ax2.set_title('Total Latency per Query')
ax2.set_ylabel('Latency (ms)')
ax2.set_xlabel('Query')
ax2.set_xticks(x)
ax2.set_xticklabels([f"Q{i+1}" for i in range(len(faiss_per_query))], rotation=45)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# --- Plot 3: Breakdown (Embedding vs Search) ---
ax3 = axes[2]
categories = ['FAISS', 'ChromaDB']
embed_means = [df_bench['faiss_embed_ms'].mean(), df_bench['chroma_embed_ms'].mean()]
search_means = [df_bench['faiss_search_ms'].mean(), df_bench['chroma_search_ms'].mean()]

ax3.bar(categories, embed_means, label='Embedding', color='#FFD700', alpha=0.8)
ax3.bar(categories, search_means, bottom=embed_means, label='HNSW Search', color=['#4A90D9', '#50C878'], alpha=0.8)
ax3.set_title('Latency Breakdown')
ax3.set_ylabel('Latency (ms)')
ax3.legend()
ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Cell 12 - Result Overlap Analysis

How much do FAISS and ChromaDB agree? If both return the same chunks for a query,
it validates the HNSW implementation. If they differ, one may have better recall.

In [ ]:
print("\n" + "="*80)
print("  RESULT OVERLAP ANALYSIS: Do FAISS and ChromaDB return the same chunks?")
print("="*80)

overlap_data = []

for q_info in benchmark_queries:
    query = q_info['query']
    filters = q_info['filters']
    
    faiss_r = search_faiss(query, k=K, filters=filters)
    chroma_r = search_chroma(query, k=K, filters=filters)
    
    faiss_ids = set(faiss_r['ids'])
    chroma_ids = set(chroma_r['ids'])
    
    common = faiss_ids & chroma_ids
    overlap_pct = len(common) / max(len(faiss_ids | chroma_ids), 1) * 100
    
    overlap_data.append({
        'Query': query[:50],
        'Filters': str(filters)[:30] if filters else 'None',
        'FAISS Results': len(faiss_r['ids']),
        'Chroma Results': len(chroma_r['ids']),
        'Common': len(common),
        'Overlap %': f"{overlap_pct:.0f}%",
    })

df_overlap = pd.DataFrame(overlap_data)
print(df_overlap.to_string(index=False))

avg_overlap = np.mean([float(d['Overlap %'].strip('%')) for d in overlap_data])
print(f"\n  Average Result Overlap: {avg_overlap:.1f}%")
if avg_overlap > 80:
    print("  [OK] High overlap -- both backends produce very similar results.")
elif avg_overlap > 50:
    print("  [WARN] Moderate overlap -- some differences in ranking behavior.")
else:
    print("  [ISSUE] Low overlap -- significant differences between backends.")

## Cell 13 - Filtered vs Unfiltered Latency Impact

Does adding metadata filters slow things down? Let's measure the overhead.

In [ ]:
test_query = "What is the structure of an atom and its properties?"
filter_configs = [
    ("No filter", None),
    ("Subject only", {"subject": "Physics"}),
    ("Subject + Class", {"subject": "Physics", "class": "12"}),
    ("Content type", {"content_type": "theory"}),
    ("Subject + Type", {"subject": "Chemistry", "content_type": "definition"}),
]

print(f"Query: {test_query}\n")

filter_bench = []
for label, filt in filter_configs:
    faiss_times = []
    chroma_times = []
    
    for _ in range(5):
        fr = search_faiss(test_query, k=5, filters=filt)
        cr = search_chroma(test_query, k=5, filters=filt)
        faiss_times.append(fr['latency_ms']['total'])
        chroma_times.append(cr['latency_ms']['total'])
    
    filter_bench.append({
        'Filter Config': label,
        'FAISS Avg (ms)': f"{np.mean(faiss_times):.2f}",
        'FAISS Std': f"{np.std(faiss_times):.2f}",
        'Chroma Avg (ms)': f"{np.mean(chroma_times):.2f}",
        'Chroma Std': f"{np.std(chroma_times):.2f}",
    })

df_filter = pd.DataFrame(filter_bench)
print("\n" + "="*80)
print("  FILTER OVERHEAD ANALYSIS")
print("="*80)
print(df_filter.to_string(index=False))

## Cell 14 - Interactive Query Testing

Use this cell to test your own queries! Change `query` and `filters` below.

In [ ]:
# ===== CUSTOMIZE YOUR QUERY HERE =====
query = "What is the photoelectric effect and who discovered it?"
filters = {"subject": "Physics"}  # Set to None for unfiltered search
k = 5  # Number of results
# =====================================

print(f"Query: {query}")
print(f"Filters: {filters}")
print(f"Top-K: {k}")

results = search_both(query, k=k, filters=filters)

display_results(results['faiss'], 'FAISS HNSW')
display_results(results['chroma'], 'ChromaDB HNSW')

# Quick latency comparison
f_ms = results['faiss']['latency_ms']['total']
c_ms = results['chroma']['latency_ms']['total']
print(f"\nFAISS: {f_ms:.2f} ms  |  ChromaDB: {c_ms:.2f} ms  |  Faster: {'FAISS' if f_ms < c_ms else 'ChromaDB'} by {abs(f_ms - c_ms):.2f} ms")

## Cell 15 - Final Efficiency Summary

A comprehensive comparison of both vector database backends.

In [ ]:
import os

# Get index sizes
faiss_size_mb = os.path.getsize(FAISS_INDEX_PATH) / (1024 * 1024)

chroma_size_mb = 0
for root, dirs, files in os.walk(CHROMA_DB_PATH):
    for f in files:
        chroma_size_mb += os.path.getsize(os.path.join(root, f))
chroma_size_mb /= (1024 * 1024)

print("\n" + "="*80)
print("  FINAL EFFICIENCY COMPARISON")
print("="*80)

comparison = pd.DataFrame({
    'Metric': [
        'Index Type',
        'Total Vectors',
        'Embedding Dimension',
        'Disk Size (MB)',
        'Native Metadata Filtering',
        'Avg Search Latency (ms)',
        'Median Search Latency (ms)',
        'P95 Search Latency (ms)',
        'Avg Total Latency (ms)',
    ],
    'FAISS HNSW': [
        'IndexHNSWFlat (M=32)',
        f"{faiss_index.ntotal:,}",
        str(faiss_index.d),
        f"{faiss_size_mb:.1f}",
        'No (post-filter)',
        f"{df_bench['faiss_search_ms'].mean():.2f}",
        f"{df_bench['faiss_search_ms'].median():.2f}",
        f"{df_bench['faiss_search_ms'].quantile(0.95):.2f}",
        f"{df_bench['faiss_total_ms'].mean():.2f}",
    ],
    'ChromaDB HNSW': [
        'HNSW (cosine space)',
        f"{collection.count():,}",
        '384',
        f"{chroma_size_mb:.1f}",
        'Yes (where clause)',
        f"{df_bench['chroma_search_ms'].mean():.2f}",
        f"{df_bench['chroma_search_ms'].median():.2f}",
        f"{df_bench['chroma_search_ms'].quantile(0.95):.2f}",
        f"{df_bench['chroma_total_ms'].mean():.2f}",
    ],
})

print(comparison.to_string(index=False))

print(f"\n{'-'*80}")
print("  KEY FINDINGS:")
print(f"{'-'*80}")
print(f"  * FAISS is a pure vector search engine -- faster raw HNSW but no built-in metadata filtering")
print(f"  * ChromaDB wraps HNSW with metadata storage -- supports native pre-filtering via 'where' clauses")
print(f"  * For latency-critical applications without complex filters -> FAISS is recommended")
print(f"  * For feature-rich RAG with metadata filters -> ChromaDB offers better developer experience")
print(f"  * Disk usage: FAISS ({faiss_size_mb:.1f} MB) vs ChromaDB ({chroma_size_mb:.1f} MB)")
print(f"{'-'*80}")